# ViT-S/16 en papa — entrenamiento en Colab (T4)
Proyecto final MIA-203. Corre **ViT-S/16 preentrenado** con 3 semillas sobre la **misma partición** que se usa localmente (`data/splits/papa_split.csv`).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.

Los resultados se guardan en **Google Drive** (`MyDrive/MIA203/results`), así que si Colab se desconecta, al volver a correr se **saltan las semillas ya terminadas**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Montar Drive y traer el paquete del proyecto
Sube `colab_bundle.zip` a `MyDrive/MIA203/` (recomendado) o súbelo cuando la celda lo pida.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/MIA203'
os.makedirs(DRIVE, exist_ok=True)
bundle = f'{DRIVE}/colab_bundle.zip'
if not os.path.exists(bundle):
    from google.colab import files
    print('No encontré colab_bundle.zip en Drive: súbelo ahora')
    up = files.upload()
    shutil.move(list(up)[0], bundle)

PROY = '/content/proyecto'
shutil.rmtree(PROY, ignore_errors=True)
zipfile.ZipFile(bundle).extractall(PROY)
os.chdir(PROY)
!ls

## 2. Dependencias

In [ ]:
!pip -q install timm==1.0.30
import torch, timm, torchvision
print(torch.__version__, torchvision.__version__, timm.__version__, torch.cuda.get_device_name(0))

## 3. Verificar que la partición es la misma que la local

In [ ]:
import pandas as pd
df = pd.read_csv('data/splits/papa_split.csv')
print(df.split.value_counts().to_dict())          # esperado: train 2112, val 450, test 453
assert df.split.value_counts().to_dict() == {'train': 2112, 'test': 453, 'val': 450}
faltan = [p for p in df.path if not os.path.exists(p)]
print('imágenes faltantes:', len(faltan))
assert not faltan
print(pd.crosstab(df.label, df.split))

## 4. Entrenar ViT-S/16 preentrenado — 3 semillas
Cada semilla guarda en Drive. Si se corta, vuelve a correr esta celda: retoma desde la siguiente semilla pendiente.

In [ ]:
RES = f'{DRIVE}/results'
os.makedirs(f'{RES}/logs', exist_ok=True)
for seed in [0, 1, 2]:
    run = f'papa_vit_small_pre_s{seed}'
    if os.path.exists(f'{RES}/{run}/metrics.json'):
        print(f'{run}: ya existe, se salta'); continue
    print(f'===== {run} =====')
    !python src/train.py --config configs/vit_small.yaml --seed {seed} --set num_workers=2 out_dir={RES} 2>&1 | tee {RES}/logs/{run}.log

## 5. Resumen (media ± DE)

In [ ]:
import json, glob, numpy as np
filas = []
for f in sorted(glob.glob(f'{RES}/papa_vit_small_*/metrics.json')):
    m = json.load(open(f))
    filas.append(dict(run=m['run'], seed=m['seed'], best_epoch=m['best_epoch'], min=round(m['train_time_min'], 1),
                      **{f'test_{k}': v for k, v in m['test'].items()}))
t = pd.DataFrame(filas); display(t)
for k in ['test_macro_f1', 'test_balanced_acc', 'test_accuracy']:
    print(f'{k}: {t[k].mean():.4f} ± {t[k].std(ddof=1):.4f}')

## 6. ResNet-50 preentrenada + ablación desde cero (ViT y ResNet) — 3 semillas c/u


In [ ]:
# Todo lo que falta en Colab: ResNet-50 preentrenada (3 semillas) + ablación desde cero (R4)
# Salta lo que ya exista en Drive; si Colab se corta, vuelve a ejecutar esta celda.
cola = [('resnet50', 'true', s) for s in [0, 1, 2]] +        [('vit_small', 'false', s) for s in [0, 1, 2]] +        [('resnet50', 'false', s) for s in [0, 1, 2]]
for cfg, pre, seed in cola:
    run = f"papa_{cfg}_{'pre' if pre == 'true' else 'scratch'}_s{seed}"
    if os.path.exists(f'{RES}/{run}/metrics.json'):
        print(f'{run}: ya existe, se salta'); continue
    print(f'===== {run} =====')
    !python src/train.py --config configs/{cfg}.yaml --seed {seed} --pretrained {pre} --set num_workers=2 out_dir={RES} 2>&1 | tee {RES}/logs/{run}.log

## 7. Descargar resultados
También quedan en Drive (`MyDrive/MIA203/results`). Este zip incluye los pesos `best.pt` (~85 MB por corrida).

In [ ]:
shutil.make_archive('/content/resultados_vit', 'zip', RES)
from google.colab import files
files.download('/content/resultados_vit.zip')